# Upload NF-UQ-NIDS-v2 to Google Drive

Local source: `/Volumes/LocalDrive1/Dataset/Security/NF-UQ-NIDS-v2/raw/NF-UQ-NIDS-v2.csv`

Drive destination: `gdrive:projects/Dataset/Security/NF-UQ-NIDS-v2`

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

DATASET_NAME = "NF-UQ-NIDS-v2"
LOCAL_SOURCE = Path("/Volumes/LocalDrive1/Dataset/Security/NF-UQ-NIDS-v2/raw/NF-UQ-NIDS-v2.csv")
DRIVE_DESTINATION = "gdrive:projects/Dataset/Security/NF-UQ-NIDS-v2"
EXPECTED_MIN_CSV_COUNT = 1
EXCLUDES = ["._*", ".DS_Store", "*.part"]

def ensure_kagglehub() -> None:
    try:
        import kagglehub  # noqa: F401
        return
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])


def try_download_nf_uq_nids_v2() -> Path | None:
    ensure_kagglehub()
    import kagglehub

    slugs = [
        "aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset",
        "dhoogla/nfuqnidsv2",
    ]
    for slug in slugs:
        try:
            print("NF CSV not found locally. Trying KaggleHub dataset:", slug)
            cache_path = Path(kagglehub.dataset_download(slug))
            print("KaggleHub cache:", cache_path)
            matches = list(cache_path.rglob("NF-UQ-NIDS-v2.csv")) if cache_path.is_dir() else [cache_path]
            for match in matches:
                if match.exists() and match.is_file() and match.stat().st_size > 1_000_000:
                    return match
        except Exception as exc:
            print("KaggleHub failed for", slug, "->", exc)
    return None


def find_nf_uq_nids_csv() -> Path:
    candidates = []
    env_path = os.environ.get("NF_UQ_NIDS_V2_CSV")
    if env_path:
        candidates.append(Path(env_path))
    candidates.extend([
        LOCAL_SOURCE,
        Path.home() / ".cache" / "kagglehub" / "datasets" / "aryashah2k" / "nfuqnidsv2-network-intrusion-detection-dataset" / "versions" / "1" / "NF-UQ-NIDS-v2.csv",
        Path("/Users/darsh/.cache/kagglehub/datasets/aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset/versions/1/NF-UQ-NIDS-v2.csv"),
    ])
    search_roots = [
        Path("/Volumes/LocalDrive1/Dataset/Security/NF-UQ-NIDS-v2"),
        Path.home() / ".cache" / "kagglehub" / "datasets",
        Path("/Users/darsh/.cache/kagglehub/datasets"),
    ]
    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob("NF-UQ-NIDS-v2.csv"))
    for candidate in candidates:
        if candidate.exists() and candidate.is_file() and candidate.stat().st_size > 1_000_000:
            return candidate
    downloaded = try_download_nf_uq_nids_v2()
    if downloaded is not None:
        return downloaded
    checked = "\n".join(str(p) for p in candidates[:20])
    raise FileNotFoundError(
        "Missing NF-UQ-NIDS-v2 CSV and KaggleHub fallback failed. Run the NF-UQ-NIDS-v2 download notebook first, "
        "or set os.environ['NF_UQ_NIDS_V2_CSV'] to the actual CSV path before this cell. Checked:\n"
        f"{checked}"
    )

def find_rclone() -> str:
    env_path = os.environ.get("RCLONE_BIN")
    if env_path and Path(env_path).exists():
        return env_path
    found = shutil.which("rclone")
    if found:
        return found
    candidates = ["/opt/homebrew/bin/rclone", "/usr/local/bin/rclone", "/opt/local/bin/rclone"]
    candidates.extend(str(p) for p in Path("/opt/homebrew/Cellar/rclone").glob("*/bin/rclone"))
    for candidate in candidates:
        if Path(candidate).exists():
            return candidate
    for shell in ["/bin/zsh", "/usr/bin/zsh", "/bin/bash", "/usr/bin/bash", "/bin/sh", "/usr/bin/sh"]:
        if not Path(shell).exists():
            continue
        try:
            shell_found = subprocess.run([shell, "-lc", "command -v rclone"], capture_output=True, text=True)
        except OSError:
            continue
        if shell_found.returncode == 0 and shell_found.stdout.strip():
            return shell_found.stdout.strip()
    checked = ", ".join(candidates)
    raise RuntimeError(
        "rclone is not available in this runtime. Run this on the local Mac runtime with rclone configured. "
        "If 01 printed a working path, set it here first: os.environ['RCLONE_BIN'] = '/that/path'. "
        f"Checked: {checked}"
    )


LOCAL_SOURCE = find_nf_uq_nids_csv()
print("Resolved local source:", LOCAL_SOURCE)

try:
    RCLONE = find_rclone()
    UPLOAD_MODE = "rclone"
    DRIVE_DESTINATION_LOCAL = None
    print("rclone:", RCLONE)
except RuntimeError as exc:
    RCLONE = None
    UPLOAD_MODE = "mounted_drive"
    DRIVE_DESTINATION_LOCAL = Path("/content/drive/MyDrive/projects/Dataset/Security/NF-UQ-NIDS-v2")
    print("rclone unavailable; using mounted Google Drive fallback.")
    print(exc)

csv_files = [LOCAL_SOURCE] if LOCAL_SOURCE.name.endswith(".csv") and not LOCAL_SOURCE.name.startswith("._") else []
print("Dataset:", DATASET_NAME)
print("Local source:", LOCAL_SOURCE)
print("Source is symlink:", LOCAL_SOURCE.is_symlink())
print("CSV count:", len(csv_files))
print("File size bytes:", LOCAL_SOURCE.stat().st_size)
print("Drive destination:", DRIVE_DESTINATION)
print("Upload mode:", UPLOAD_MODE)
if len(csv_files) < EXPECTED_MIN_CSV_COUNT:
    raise RuntimeError(f"Expected at least {EXPECTED_MIN_CSV_COUNT} CSV files, found {len(csv_files)}")

NF CSV not found locally. Trying KaggleHub dataset: aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset
Using Colab cache for faster access to the 'nfuqnidsv2-network-intrusion-detection-dataset' dataset.
KaggleHub cache: /kaggle/input/nfuqnidsv2-network-intrusion-detection-dataset
Resolved local source: /kaggle/input/nfuqnidsv2-network-intrusion-detection-dataset/NF-UQ-NIDS-v2.csv
rclone unavailable; using mounted Google Drive fallback.
rclone is not available in this runtime. Run this on the local Mac runtime with rclone configured. If 01 printed a working path, set it here first: os.environ['RCLONE_BIN'] = '/that/path'. Checked: /opt/homebrew/bin/rclone, /usr/local/bin/rclone, /opt/local/bin/rclone
Dataset: NF-UQ-NIDS-v2
Local source: /kaggle/input/nfuqnidsv2-network-intrusion-detection-dataset/NF-UQ-NIDS-v2.csv
Source is symlink: False
CSV count: 1
File size bytes: 13729330230
Drive destination: gdrive:projects/Dataset/Security/NF-UQ-NIDS-v2
Upload mode: mounted_drive


In [2]:
if UPLOAD_MODE == "rclone":
    cmd = [
        RCLONE, "copy", str(LOCAL_SOURCE), DRIVE_DESTINATION,
        "--copy-links",
        "--progress",
        "--transfers", "4",
        "--checkers", "8",
        "--drive-chunk-size", "256M",
        "--stats", "30s",
    ]
    for pattern in EXCLUDES:
        cmd.extend(["--exclude", pattern])
    print(" ".join(cmd))
    subprocess.check_call(cmd)
else:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "rclone is unavailable and google.colab.drive is unavailable. "
            "Run this notebook in Colab with Drive mount support, or use the local Mac runtime with rclone."
        ) from exc

    drive.mount("/content/drive")
    DRIVE_DESTINATION_LOCAL.mkdir(parents=True, exist_ok=True)
    destination_file = DRIVE_DESTINATION_LOCAL / LOCAL_SOURCE.name
    source_size = LOCAL_SOURCE.stat().st_size
    if destination_file.exists() and destination_file.stat().st_size == source_size:
        print("Destination already has matching file:", destination_file)
    else:
        print("Copying to:", destination_file)
        chunk_size = 64 * 1024 * 1024
        copied = 0
        with LOCAL_SOURCE.open("rb") as src, destination_file.open("wb") as dst:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                copied += len(chunk)
                if copied == source_size or copied % (1024 * 1024 * 1024) < chunk_size:
                    print(f"Copied {copied / (1024 ** 3):.2f} GiB / {source_size / (1024 ** 3):.2f} GiB")
        print("Copy complete:", destination_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying to: /content/drive/MyDrive/projects/Dataset/Security/NF-UQ-NIDS-v2/NF-UQ-NIDS-v2.csv
Copied 1.00 GiB / 12.79 GiB
Copied 2.00 GiB / 12.79 GiB
Copied 3.00 GiB / 12.79 GiB
Copied 4.00 GiB / 12.79 GiB
Copied 5.00 GiB / 12.79 GiB
Copied 6.00 GiB / 12.79 GiB
Copied 7.00 GiB / 12.79 GiB
Copied 8.00 GiB / 12.79 GiB
Copied 9.00 GiB / 12.79 GiB
Copied 10.00 GiB / 12.79 GiB
Copied 11.00 GiB / 12.79 GiB
Copied 12.00 GiB / 12.79 GiB
Copied 12.79 GiB / 12.79 GiB
Copy complete: /content/drive/MyDrive/projects/Dataset/Security/NF-UQ-NIDS-v2/NF-UQ-NIDS-v2.csv


In [3]:
if UPLOAD_MODE == "rclone":
    verify_cmd = [RCLONE, "size", DRIVE_DESTINATION, "--exclude", "._*", "--exclude", ".DS_Store"]
    print(" ".join(verify_cmd))
    subprocess.check_call(verify_cmd)
else:
    destination_file = DRIVE_DESTINATION_LOCAL / LOCAL_SOURCE.name
    if not destination_file.exists():
        raise FileNotFoundError(f"Upload verification failed. Missing: {destination_file}")
    print("Verified Drive file:", destination_file)
    print("Bytes:", destination_file.stat().st_size)
    if destination_file.stat().st_size != LOCAL_SOURCE.stat().st_size:
        raise RuntimeError("Upload verification failed: destination size does not match source size.")

Verified Drive file: /content/drive/MyDrive/projects/Dataset/Security/NF-UQ-NIDS-v2/NF-UQ-NIDS-v2.csv
Bytes: 13729330230
